# Data Contract And Lineage

Proves the exact generated operational dataset, row counts, stable hash, source boundary and relational coverage.

In [1]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

ROOT = Path.cwd()
if not (ROOT / "outputs").exists():
    ROOT = Path("Ai miroservices/modeling/project_operational_baseline").resolve()
OUT = ROOT / "outputs"
EVAL = OUT / "evaluator"
sns.set_theme(style="whitegrid")

In [2]:
manifest = json.loads((OUT / 'manifest.json').read_text())
manifest

{'dataset_version': 'PROJECT_OPERATIONAL_BASELINE_V3',
 'quality_tier': 'GENERATED_OPERATIONAL_BASELINE',
 'seed': 20260715,
 'dataset_hash': 'ba12a1d46e221a5feefa890e10976ef0df76493dab75e1de7eeffd327b38aa22',
 'config': {'seed': 20260715,
  'history_months': 72,
  'operational_months': 18,
  'fg_count': 16,
  'rm_count': 48,
  'pm_count': 32,
  'location_count': 600,
  'supplier_count': 16,
  'customer_count': 48,
  'worker_count': 24,
  'order_count': 5000,
  'order_line_count': 15000,
  'stock_movement_count': 30000,
  'task_count': 30000,
  'start_month': '2020-01-01'},
 'row_counts': {'warehouses': 1,
  'locations': 606,
  'warehouse_graph_nodes': 52,
  'warehouse_graph_edges': 55,
  'materials': 80,
  'finished_goods': 16,
  'bom_components': 250,
  'production_history': 1152,
  'demand_history': 5760,
  'classification_thresholds': 16,
  'material_classifications': 80,
  'inventory_policy': 80,
  'inventory': 169,
  'suppliers': 16,
  'supplier_materials': 80,
  'customers': 48,

In [3]:
pd.DataFrame({'table': manifest['row_counts'].keys(), 'rows': manifest['row_counts'].values()}).sort_values('rows', ascending=False)

,table,rows
21,operation_events,30000
20,tasks,30000
19,stock_movements,30000
18,order_items,15000
8,demand_history,5760
17,orders,5000
7,production_history,1152
1,locations,606
6,bom_components,250
12,inventory,169


In [4]:
materials = pd.read_csv(OUT / 'materials.csv.gz')
materials.info()
display(materials.head())
display(materials.groupby(['material_type','category']).size().rename('rows').reset_index())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 80 entries, 0 to 79
Data columns (total 34 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   material_id                80 non-null     object 
 1   material_code              80 non-null     object 
 2   description                80 non-null     object 
 3   source_item_code           80 non-null     object 
 4   source_record_type         80 non-null     object 
 5   material_type              80 non-null     object 
 6   category                   80 non-null     object 
 7   unit_type                  80 non-null     object 
 8   formula_role               80 non-null     object 
 9   recipe_groups              48 non-null     object 
 10  target_fg_indexes          32 non-null     object 
 11  storage_type               80 non-null     object 
 12  handling_unit_type         80 non-null     object 
 13  units_per_handling_unit    80 non-null     int64  
 

,material_id,material_code,description,source_item_code,source_record_type,material_type,category,unit_type,formula_role,recipe_groups,...,units_per_pallet,pallet_spaces,max_pallet_weight_kg,stackable,max_stack_height,temperature_controlled,hazardous,fragile,shelf_life_days,data_quality_tier
0,c744dcbe-24e0-5790-8b9c-a54dca036f56,RM-0001,CAUSTIC SODA,100036,SUPPLIED_ITEM_MASTER,raw_material,BAG,KG,ALKALI,SOAP|HOME,...,1080,1,1020.0,False,1,False,True,False,1095,GENERATED_OPERATIONAL_BASELINE
1,460ce32d-2531-52a4-9ff8-d6187cbdf8eb,RM-0002,CALCIUM CARBONATE (GROUND),101054,SUPPLIED_ITEM_MASTER,raw_material,BAG,KG,FILLER,ORAL|HOME,...,800,1,1020.0,True,2,False,False,False,1095,GENERATED_OPERATIONAL_BASELINE
2,99a79093-e65d-5dce-9a6f-697e975e6301,RM-0003,SORBITOL,100098,SUPPLIED_ITEM_MASTER,raw_material,DRUM,KG,HUMECTANT,SOAP|SKIN|ORAL,...,704,1,780.0,True,3,False,False,False,1095,GENERATED_OPERATIONAL_BASELINE
3,cd647bd8-f185-5b06-a747-47d01c85fbcf,RM-0004,TALCUM POWDER,100108,SUPPLIED_ITEM_MASTER,raw_material,BAG,KG,FILLER,SKIN,...,1320,1,1020.0,True,3,False,False,False,730,GENERATED_OPERATIONAL_BASELINE
4,4f3a7839-cc51-58b2-b5b3-acd2b4c5ca4f,RM-0005,SILICA 165 THICKENING,100094,SUPPLIED_ITEM_MASTER,raw_material,BAG,KG,THICKENER,LIQUID_WASH|ORAL,...,1160,1,1020.0,True,4,False,False,False,730,GENERATED_OPERATIONAL_BASELINE


,material_type,category,rows
0,packaging_material,BOTTLE,4
1,packaging_material,CAP,3
2,packaging_material,CARTON,5
3,packaging_material,CORR_BOX,4
4,packaging_material,INNER_LINER,1
5,packaging_material,INSERT,7
6,packaging_material,JAR,1
7,packaging_material,NECK_TAG,1
8,packaging_material,POUCH,3
9,packaging_material,TUBE,2


**Claim boundary.** These rows are the permanent project-operational baseline because external customer history is unavailable. They are suitable for reproducible system validation, but they are not externally observed customer records.